# TASK 3: Customer Cohort, Churn & Retention Intelligence
### Diagnostic Cross-Sectional Cohort Analysis, High-Risk Segmentation, and Predictive Retention Modeling

**Author**: Vijay Maheswari  
**Framework**: RISK Methodology — Intelligence Extraction & Solution Development  
**Dataset**: Telco Customer Churn (7,043 validated subscriber profiles)


In [1]:
import sys
import os

for p in [os.path.abspath(".."), os.path.abspath(".")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from src.loaders import load_telco_churn
from src.cleaning import clean_telco_churn
from src.metrics import calculate_churn_kpis
from src.validation import validate_churn_schema

sns.set_theme(style="whitegrid", palette="muted")


## 1. Dataset Ingestion & Quality Audit
We validate customer identifiers, resolve blank TotalCharges for tenure=0 new subscribers, and create standardized 12-month tenure cohorts.


In [2]:
raw_df = load_telco_churn()
print(f"Raw churn records: {raw_df.shape}")

# Quality audit
audit = validate_churn_schema(raw_df)
print(audit.summary())

# Clean and transform
df = clean_telco_churn(raw_df)
print(f"Cleaned churn records: {df.shape}")


Raw churn records: (7043, 21)
[PASSED] Telco Customer Churn: 7043 rows, 21 cols | Warnings: 1


Cleaned churn records: (7043, 25)


## 2. Core Churn & Retention Scorecard
We calculate benchmark churn rate, retention rate, and cohort breakdown.


In [3]:
kpis = calculate_churn_kpis(df)
print(f"Eligible Customers : {kpis['Eligible Customers']:,}")
print(f"Churned Customers  : {kpis['Churned Customers']:,} ({kpis['Churn Rate %']}%)")
print(f"Retained Customers : {kpis['Retained Customers']:,} ({kpis['Retention Rate %']}%)")
print(f"High-Risk Segment  : {kpis['High Risk Cohort Total']:,} customers with {kpis['High Risk Churn Rate %']}% Churn Rate")


Eligible Customers : 7,043
Churned Customers  : 1,869 (26.54%)
Retained Customers : 5,174 (73.46%)
High-Risk Segment  : 1,307 customers with 60.37% Churn Rate


## 3. Tenure Cohort Retention Table
Analyzing churn across 12-month tenure bins highlights Year 1 vulnerability.


In [4]:
kpis["Tenure Cohorts"]


,Tenure Cohort,Total,Churned,Retained,Churn Rate %,Retention Rate %
0,0-12m,2186,1037,1149,47.438243,52.561757
1,13-24m,1024,294,730,28.710938,71.289062
2,25-36m,832,180,652,21.634615,78.365385
3,37-48m,762,145,617,19.028871,80.971129
4,49-60m,832,120,712,14.423077,85.576923
5,61-72m,1407,93,1314,6.609808,93.390192


## 4. Contract & Payment Breakdown
Evaluating contract stability and payment friction.


In [5]:
print("Contract Breakdown:")
print(kpis["Contract Breakdown"])
print("Payment Method Breakdown:")
print(kpis["Payment Breakdown"])


Contract Breakdown:
         Contract  Total  Churned  Churn Rate %
0  Month-to-month   3875     1655     42.709677
1        One year   1473      166     11.269518
2        Two year   1695       48      2.831858
Payment Method Breakdown:
               PaymentMethod  Total  Churned  Churn Rate %
0  Bank transfer (automatic)   1544      258     16.709845
1    Credit card (automatic)   1522      232     15.243101
2           Electronic check   2365     1071     45.285412
3               Mailed check   1612      308     19.106700


## 5. Visual Diagnostics Generation
Render and export all 5 churn diagnostic and predictive figures.


In [6]:
from src.generate_churn_intelligence import generate_churn_visualizations

generate_churn_visualizations(df, "outputs/figures")
print("Visualizations saved to outputs/figures/")


Visualizations saved to outputs/figures/


## 6. Predictive Classification Modeling & Feature Attribution
We train a calibrated Logistic Regression classifier to predict subscriber churn likelihood and prevent target leakage.


In [7]:
feature_cols = [
    "tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen",
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod"
]

X = pd.get_dummies(df[feature_cols], drop_first=True)
y = df["Churn_Numeric"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred = lr.predict(X_test_scaled)
y_prob = lr.predict_proba(X_test_scaled)[:, 1]

print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

# Top Risk & Retention Drivers
coefs = pd.Series(lr.coef_[0], index=X.columns).sort_values()
top_drivers = pd.concat([coefs.head(5), coefs.tail(5)])
print("Top 5 Retention Drivers (Negative) and Top 5 Churn Drivers (Positive):")
print(top_drivers)


ROC-AUC Score: 0.8418
Classification Report:
              precision    recall  f1-score   support

    Retained       0.85      0.89      0.87      1035
     Churned       0.66      0.57      0.61       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

Top 5 Retention Drivers (Negative) and Top 5 Churn Drivers (Positive):
tenure                        -1.236528
MonthlyCharges                -0.920153
Contract_Two year             -0.586859
Contract_One year             -0.285509
OnlineSecurity_Yes            -0.123561
MultipleLines_Yes              0.216167
StreamingTV_Yes                0.257144
StreamingMovies_Yes            0.257227
TotalCharges                   0.514285
InternetService_Fiber optic    0.776154
dtype: float64
